# Bit Flip And Phase Flip Error Correction

Commented revision of `qc-lab5.ipynb`, prepared 2026-11-06. Run cells from top to bottom. Qiskit displays classical bit strings with the highest-index bit on the left. Original import copies have outputs cleared and embedded credentials removed.

Fixes: replace the undefined `corrected_qc` variable with measured conditional corrections and verify recovery for each data qubit. Phase errors are converted to bit errors by a basis change before applying the same syndrome decoder. Assumes at most one error per repetition block.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.quantum_info import state_fidelity, Statevector
from IPython.display import display

# Two parity checks identify one error among three data qubits.
# Display order c[1]c[0]: 01 -> q0, 11 -> q1, 10 -> q2.
def correct_repetition(error_qubit, phase=False, psi=(0.6, 0.8)):
    qc = QuantumCircuit(5, 2)
    qc.initialize(psi, 0)
    qc.cx(0, 1)
    qc.cx(0, 2)
    if phase:
        # Hadamards turn the bit repetition code into a phase repetition code.
        qc.h([0, 1, 2])
        qc.z(error_qubit)
        qc.h([0, 1, 2])  # Convert the Z error into an X error for detection.
    else:
        qc.x(error_qubit)
    qc.cx(0, 3)
    qc.cx(1, 3)
    qc.cx(1, 4)
    qc.cx(2, 4)
    qc.measure(3, 0)
    qc.measure(4, 1)
    for syndrome, target in [(1, 0), (3, 1), (2, 2)]:
        with qc.if_test((qc.cregs[0], syndrome)):
            qc.x(target)
    # Decode in the computational basis, recovering the input on q0.
    qc.cx(0, 2)
    qc.cx(0, 1)
    qc.save_density_matrix([0])
    backend = AerSimulator(method="density_matrix")
    from qiskit import transpile
    result = backend.run(transpile(qc, backend), shots=16, seed_simulator=42).result()
    fidelity = state_fidelity(result.data(0)["density_matrix"], Statevector(list(psi)))
    assert np.isclose(fidelity, 1.0), (phase, error_qubit, fidelity)
    return qc, fidelity

# Exercise every possible single error location for both codes.
for phase in [False, True]:
    for error_qubit in range(3):
        circuit, fidelity = correct_repetition(error_qubit, phase)
        print("Phase flip" if phase else "Bit flip", error_qubit, "fidelity:", fidelity)
display(circuit.draw("mpl"))